# Agent 2: Independent Random Forest Models

This notebook trains two separate models from the project's pre-split cleaned data:

- `FraudRandomForest` reads only `clean_data/cc_data.pkl` and predicts `Class` (1 = fraudulent transaction).
- `HomeCreditRandomForest` reads only `clean_data/hc_data.pkl` and predicts `TARGET` (1 = payment difficulties).

Each agent owns its data loader, preprocessing pipeline, classifier, held-out evaluation, model artifact, and output directory. The two datasets are never concatenated or shared between models.

The pickle files already contain stratified train/test partitions. The upstream cleaning notebook performs some scaling, imputation, and category encoding before or around splitting; this notebook cannot undo any leakage already baked into these pickles. It keeps the supplied split intact and fits the Random Forest imputation and one-hot encoding stages on each training partition only. Home Credit's pickle stores 13 categorical fields as numeric label codes, so the model treats those fields as categories and one-hot encodes them. Prediction inputs for those fields must use the same numeric codes as the cleaned pickle; the pickle does not include the original label-encoder mappings.

In [1]:
import json
from collections.abc import Mapping
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
N_ESTIMATORS = 300


def find_project_root():
    possible_roots = [Path.cwd(), *Path.cwd().parents]
    possible_roots.append(
        Path.home() / "OneDrive" / "Desktop" / "randomgorest" / "credit-scoring-project"
    )
    for possible_root in possible_roots:
        if (possible_root / "clean_data").is_dir():
            return possible_root
    raise FileNotFoundError(
        "Could not locate the project directory containing clean_data/. "
        "Run this notebook from the repository or set PROJECT_ROOT explicitly."
    )


PROJECT_ROOT = find_project_root()
CLEAN_DATA_DIR = PROJECT_ROOT / "clean_data"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
print("Project directory:", PROJECT_ROOT)

Project directory: c:\Users\Mer_Hass\OneDrive\Desktop\randomgorest\credit-scoring-project


In [2]:
class IndependentRandomForestAgent:
    def __init__(
        self,
        agent_name,
        dataset_type,
        dataset_file,
        target_column,
        identifier_columns,
        model_filename,
        output_folder,
        class_labels,
    ):
        self.agent_name = agent_name
        self.dataset_type = dataset_type
        self.dataset_file = dataset_file
        self.target_column = target_column
        self.identifier_columns = identifier_columns
        self.model_filename = model_filename
        self.model_name = Path(model_filename).stem
        self.output_dir = OUTPUTS_DIR / output_folder
        self.model_path = MODELS_DIR / model_filename
        self.class_labels = class_labels
        self._split = None
        self.pipeline = None
        self.metrics = None
        self.feature_columns = None

    def load_data(self):
        raise NotImplementedError

    @staticmethod
    def _as_frame(values, feature_names=None):
        if isinstance(values, pd.DataFrame):
            frame = values.copy()
        elif isinstance(values, np.ndarray):
            frame = pd.DataFrame(values, columns=feature_names)
        else:
            frame = pd.DataFrame(values)
        frame.columns = [str(column) for column in frame.columns]
        return frame.reset_index(drop=True)

    def get_split(self):
        if self._split is not None:
            return self._split

        raw_train, raw_test, raw_target_train, raw_target_test = self.load_data()
        features_train = self._as_frame(raw_train)
        features_test = self._as_frame(raw_test, feature_names=features_train.columns)
        target_train = pd.Series(raw_target_train, name=self.target_column).reset_index(drop=True)
        target_test = pd.Series(raw_target_test, name=self.target_column).reset_index(drop=True)

        if len(features_train) != len(target_train) or len(features_test) != len(target_test):
            raise ValueError(f"{self.dataset_type}: feature and target row counts do not match.")

        for split_name, features, target in (
            ("train", features_train, target_train),
            ("test", features_test, target_test),
        ):
            missing_target = target.isna()
            if missing_target.any():
                print(f"{self.dataset_type}: removing {int(missing_target.sum())} {split_name} rows with missing targets.")
                features.drop(index=target.index[missing_target], inplace=True)
                target.drop(index=target.index[missing_target], inplace=True)
                features.reset_index(drop=True, inplace=True)
                target.reset_index(drop=True, inplace=True)

        forbidden_columns = {self.target_column, *self.identifier_columns}
        columns_to_remove = [column for column in features_train.columns if column in forbidden_columns]
        if columns_to_remove:
            print(f"{self.dataset_type}: excluding target/identifier columns: {columns_to_remove}")
            features_train = features_train.drop(columns=columns_to_remove)
            features_test = features_test.drop(columns=columns_to_remove, errors="ignore")

        if set(features_train.columns) != set(features_test.columns):
            missing_test = sorted(set(features_train.columns) - set(features_test.columns))
            unexpected_test = sorted(set(features_test.columns) - set(features_train.columns))
            raise ValueError(
                f"{self.dataset_type}: train/test feature columns differ. "
                f"Missing in test: {missing_test}; unexpected in test: {unexpected_test}."
            )
        features_test = features_test.loc[:, features_train.columns]

        empty_columns = features_train.columns[features_train.isna().all()].tolist()
        if empty_columns:
            print(f"{self.dataset_type}: excluding all-missing training columns: {empty_columns}")
            features_train = features_train.drop(columns=empty_columns)
            features_test = features_test.drop(columns=empty_columns)

        expected_classes = set(self.class_labels)
        if set(pd.unique(target_train)) != expected_classes:
            raise ValueError(
                f"{self.dataset_type}: expected training classes {expected_classes}, "
                f"found {set(pd.unique(target_train))}."
            )
        if set(pd.unique(target_test)) != expected_classes:
            raise ValueError(
                f"{self.dataset_type}: expected test classes {expected_classes}, "
                f"found {set(pd.unique(target_test))}."
            )

        self.feature_columns = features_train.columns.tolist()
        self._split = (features_train, features_test, target_train, target_test)
        return self._split

    def inspect_data(self):
        features_train, features_test, target_train, target_test = self.get_split()
        print(f"{self.agent_name} | {self.dataset_type} | {self.dataset_file}")
        print("Training shape:", features_train.shape, "Testing shape:", features_test.shape)
        print("Target:", self.target_column)
        print("Feature columns:", features_train.columns.tolist())
        print("Training dtypes:")
        display(features_train.dtypes.astype(str).value_counts().rename_axis("dtype").to_frame("column_count"))
        print("Training missing values (largest 20):")
        display(features_train.isna().sum().sort_values(ascending=False).head(20).to_frame("missing_count"))
        print("Training target counts:")
        display(target_train.value_counts(dropna=False).sort_index().to_frame("count"))
        print("Test target counts:")
        display(target_test.value_counts(dropna=False).sort_index().to_frame("count"))
        print("Training target proportions:")
        display(target_train.value_counts(normalize=True).sort_index().to_frame("proportion"))
        print("Test target proportions:")
        display(target_test.value_counts(normalize=True).sort_index().to_frame("proportion"))

    def _build_pipeline(self, features_train, target_train):
        numeric_columns = features_train.select_dtypes(include=["number", "bool"]).columns.tolist()
        categorical_columns = features_train.select_dtypes(
            include=["object", "category", "string"]
        ).columns.tolist()
        unsupported_columns = [
            column for column in features_train.columns
            if column not in numeric_columns and column not in categorical_columns
        ]
        if unsupported_columns:
            raise TypeError(
                f"{self.dataset_type}: unsupported feature types in {unsupported_columns}."
            )
        if not numeric_columns and not categorical_columns:
            raise ValueError(f"{self.dataset_type}: no usable feature columns remain.")

        transformers = []
        if numeric_columns:
            numeric_pipeline = Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median", keep_empty_features=True))
            ])
            transformers.append(("numeric", numeric_pipeline, numeric_columns))
        if categorical_columns:
            categorical_pipeline = Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
                ("one_hot", OneHotEncoder(handle_unknown="ignore")),
            ])
            transformers.append(("categorical", categorical_pipeline, categorical_columns))

        preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
        class_proportions = target_train.value_counts(normalize=True)
        minority_ratio = class_proportions.min() / class_proportions.max()
        class_weight = "balanced_subsample" if minority_ratio < 0.5 else None
        classifier = RandomForestClassifier(
            n_estimators=N_ESTIMATORS,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight=class_weight,
        )
        self.class_weight = class_weight
        self.numeric_columns = numeric_columns
        self.categorical_columns = categorical_columns
        return Pipeline(steps=[("preprocessor", preprocessor), ("classifier", classifier)])

    def fit_evaluate_save(self):
        features_train, features_test, target_train, target_test = self.get_split()
        self.pipeline = self._build_pipeline(features_train, target_train)
        print(
            f"Training {self.model_name}: {len(features_train):,} rows, "
            f"{features_train.shape[1]} features, class_weight={self.class_weight}"
        )
        self.pipeline.fit(features_train, target_train)

        classifier = self.pipeline.named_steps["classifier"]
        positive_index = list(classifier.classes_).index(1)
        predictions = self.pipeline.predict(features_test)
        risk_scores = self.pipeline.predict_proba(features_test)[:, positive_index]
        positive_targets = (target_test == 1).astype(int)
        report = classification_report(
            target_test,
            predictions,
            labels=self.class_labels,
            target_names=[str(label) for label in self.class_labels],
            zero_division=0,
            output_dict=True,
        )
        self.metrics = {
            "agent_name": self.agent_name,
            "dataset_type": self.dataset_type,
            "dataset_file": self.dataset_file,
            "target_column": self.target_column,
            "model_name": self.model_name,
            "n_estimators": N_ESTIMATORS,
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
            "class_weight": self.class_weight,
            "training_rows": int(len(target_train)),
            "test_rows": int(len(target_test)),
            "accuracy": float(accuracy_score(target_test, predictions)),
            "precision_class_1": float(precision_score(target_test, predictions, pos_label=1, zero_division=0)),
            "recall_class_1": float(recall_score(target_test, predictions, pos_label=1, zero_division=0)),
            "f1_class_1": float(f1_score(target_test, predictions, pos_label=1, zero_division=0)),
            "roc_auc": float(roc_auc_score(positive_targets, risk_scores)),
            "average_precision": float(average_precision_score(positive_targets, risk_scores)),
            "classification_report": report,
        }

        self.output_dir.mkdir(parents=True, exist_ok=True)
        joblib.dump(self.pipeline, self.model_path)
        with (self.output_dir / "metrics.json").open("w", encoding="utf-8") as metrics_file:
            json.dump(self.metrics, metrics_file, indent=2)
        predictions_frame = pd.DataFrame({
            "actual": target_test.to_numpy(),
            "predicted_class": predictions,
            "risk_score_class_1": risk_scores,
        })
        predictions_frame.to_csv(self.output_dir / "predictions.csv", index=False)
        report_text = classification_report(
            target_test,
            predictions,
            labels=self.class_labels,
            target_names=[str(label) for label in self.class_labels],
            zero_division=0,
        )
        (self.output_dir / "classification_report.txt").write_text(report_text, encoding="utf-8")

        confusion = confusion_matrix(target_test, predictions, labels=self.class_labels)
        figure, axis = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay(
            confusion_matrix=confusion,
            display_labels=[str(label) for label in self.class_labels],
        ).plot(ax=axis, colorbar=False, cmap="Blues")
        axis.set_title(f"{self.agent_name}: Confusion Matrix")
        figure.tight_layout()
        figure.savefig(self.output_dir / "confusion_matrix.png", dpi=150, bbox_inches="tight")
        plt.close(figure)

        from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay
        figure, axis = plt.subplots(figsize=(6, 5))
        RocCurveDisplay.from_predictions(positive_targets, risk_scores, ax=axis, name=self.model_name)
        axis.set_title(f"{self.agent_name}: ROC Curve")
        figure.tight_layout()
        figure.savefig(self.output_dir / "roc_curve.png", dpi=150, bbox_inches="tight")
        plt.close(figure)

        figure, axis = plt.subplots(figsize=(6, 5))
        PrecisionRecallDisplay.from_predictions(positive_targets, risk_scores, ax=axis, name=self.model_name)
        axis.set_title(f"{self.agent_name}: Precision-Recall Curve")
        figure.tight_layout()
        figure.savefig(self.output_dir / "precision_recall_curve.png", dpi=150, bbox_inches="tight")
        plt.close(figure)

        transformed_names = self.pipeline.named_steps["preprocessor"].get_feature_names_out()
        importances = classifier.feature_importances_
        importance_frame = (
            pd.DataFrame({"feature": transformed_names, "importance": importances})
            .sort_values("importance", ascending=False)
            .head(25)
        )
        importance_frame.to_csv(self.output_dir / "feature_importance.csv", index=False)
        figure, axis = plt.subplots(figsize=(9, 7))
        sns.barplot(data=importance_frame, x="importance", y="feature", ax=axis)
        axis.set_title(f"{self.agent_name}: Top Feature Importances")
        figure.tight_layout()
        figure.savefig(self.output_dir / "feature_importance.png", dpi=150, bbox_inches="tight")
        plt.close(figure)

        print("Saved complete pipeline:", self.model_path)
        print("Saved evaluation artifacts:", self.output_dir)
        display(pd.Series({key: value for key, value in self.metrics.items() if isinstance(value, float)}).to_frame("score"))
        print("\nClassification report:\n", report_text)
        return self.metrics

    def predict_records(self, applicant_features):
        if self.pipeline is None:
            raise RuntimeError(f"{self.model_name} has not been trained or loaded.")
        if isinstance(applicant_features, pd.DataFrame):
            input_frame = applicant_features.copy()
            single_record = len(input_frame) == 1
        elif isinstance(applicant_features, Mapping):
            input_frame = pd.DataFrame([applicant_features])
            single_record = True
        else:
            input_frame = pd.DataFrame(applicant_features)
            single_record = False
        input_frame.columns = [str(column) for column in input_frame.columns]
        input_frame = input_frame.drop(
            columns=[self.target_column, *self.identifier_columns], errors="ignore"
        )
        input_frame = input_frame.reindex(columns=self.feature_columns)
        predicted_classes = self.pipeline.predict(input_frame)
        positive_index = list(self.pipeline.named_steps["classifier"].classes_).index(1)
        risk_scores = self.pipeline.predict_proba(input_frame)[:, positive_index]
        results = [
            {
                "agent_name": self.agent_name,
                "dataset_type": self.dataset_type,
                "predicted_class": int(predicted_class),
                "risk_score": float(risk_score),
                "model_name": self.model_name,
            }
            for predicted_class, risk_score in zip(predicted_classes, risk_scores)
        ]
        return results[0] if single_record else results


class FraudRandomForest(IndependentRandomForestAgent):
    def __init__(self):
        super().__init__(
            agent_name="Agent 2 - Random Forest",
            dataset_type="credit_card_fraud",
            dataset_file="cc_data.pkl",
            target_column="Class",
            identifier_columns=[],
            model_filename="fraud_random_forest.joblib",
            output_folder="fraud_detection",
            class_labels=[0, 1],
        )

    def load_data(self):
        dataset_path = CLEAN_DATA_DIR / "cc_data.pkl"
        if not dataset_path.is_file():
            raise FileNotFoundError(f"Credit-card dataset not found: {dataset_path}")
        cleaned_data = pd.read_pickle(dataset_path)
        if not isinstance(cleaned_data, (tuple, list)) or len(cleaned_data) != 4:
            raise ValueError("cc_data.pkl must contain (X_train, X_test, y_train, y_test).")
        return cleaned_data


class HomeCreditRandomForest(IndependentRandomForestAgent):
    def __init__(self):
        super().__init__(
            agent_name="Agent 2 - Random Forest",
            dataset_type="home_credit_default",
            dataset_file="hc_data.pkl",
            target_column="TARGET",
            identifier_columns=["SK_ID_CURR"],
            model_filename="home_credit_random_forest.joblib",
            output_folder="home_credit",
            class_labels=[0, 1],
        )

    def load_data(self):
        dataset_path = CLEAN_DATA_DIR / "hc_data.pkl"
        if not dataset_path.is_file():
            raise FileNotFoundError(f"Home Credit dataset not found: {dataset_path}")
        cleaned_data = pd.read_pickle(dataset_path)
        if not isinstance(cleaned_data, (tuple, list)) or len(cleaned_data) != 5:
            raise ValueError(
                "hc_data.pkl must contain (X_train, X_test, y_train, y_test, feature_names)."
            )
        features_train, features_test, target_train, target_test, feature_names = cleaned_data
        if len(feature_names) != features_train.shape[1] or len(feature_names) != features_test.shape[1]:
            raise ValueError("hc_data.pkl feature_names do not match its feature arrays.")
        features_train = pd.DataFrame(features_train, columns=feature_names)
        features_test = pd.DataFrame(features_test, columns=feature_names)
        return features_train, features_test, target_train, target_test

In [10]:
_HomeCreditRandomForestBase = HomeCreditRandomForest


class HomeCreditRandomForest(_HomeCreditRandomForestBase):
    categorical_feature_columns = [
        "NAME_CONTRACT_TYPE",
        "CODE_GENDER",
        "FLAG_OWN_CAR",
        "FLAG_OWN_REALTY",
        "NAME_TYPE_SUITE",
        "NAME_INCOME_TYPE",
        "NAME_EDUCATION_TYPE",
        "NAME_FAMILY_STATUS",
        "NAME_HOUSING_TYPE",
        "OCCUPATION_TYPE",
        "WEEKDAY_APPR_PROCESS_START",
        "ORGANIZATION_TYPE",
        "EMERGENCYSTATE_MODE",
    ]

    def get_split(self):
        features_train, features_test, target_train, target_test = super().get_split()
        for features in (features_train, features_test):
            for column in self.categorical_feature_columns:
                if column in features.columns:
                    features[column] = features[column].astype("string")
        return features_train, features_test, target_train, target_test

    def predict_records(self, applicant_features):
        if isinstance(applicant_features, Mapping):
            input_frame = pd.DataFrame([applicant_features])
        elif isinstance(applicant_features, pd.DataFrame):
            input_frame = applicant_features.copy()
        else:
            input_frame = pd.DataFrame(applicant_features)
        for column in self.categorical_feature_columns:
            if column in input_frame.columns:
                input_frame[column] = input_frame[column].astype("string")
        return super().predict_records(input_frame)

In [3]:
fraud_agent = FraudRandomForest()
home_credit_agent = HomeCreditRandomForest()

fraud_agent.inspect_data()
home_credit_agent.inspect_data()

Agent 2 - Random Forest | credit_card_fraud | cc_data.pkl
Training shape: (227845, 30) Testing shape: (56962, 30)
Target: Class
Feature columns: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'scaled_amount', 'scaled_time']
Training dtypes:


,column_count
dtype,
float64,30


Training missing values (largest 20):


,missing_count
V1,0
V2,0
V3,0
V4,0
V5,0
V6,0
V7,0
V8,0
V9,0
V10,0


Training target counts:


,count
Class,
0,227451
1,394


... (14 outputs left out)

Training target proportions:


,proportion
TARGET,
0,0.919271
1,0.080729


Test target proportions:


,proportion
TARGET,
0,0.919272
1,0.080728


## Fraud Detection Model

`FraudRandomForest` uses only `cc_data.pkl` and writes its model to `models/fraud_random_forest.joblib`. Its metrics, predictions, report, and plots are isolated under `outputs/fraud_detection/`.

In [4]:
fraud_metrics = fraud_agent.fit_evaluate_save()

Training fraud_random_forest: 227,845 rows, 30 features, class_weight=balanced_subsample
Saved complete pipeline: c:\Users\Mer_Hass\OneDrive\Desktop\randomgorest\credit-scoring-project\models\fraud_random_forest.joblib
Saved evaluation artifacts: c:\Users\Mer_Hass\OneDrive\Desktop\randomgorest\credit-scoring-project\outputs\fraud_detection


,score
accuracy,0.999508
precision_class_1,0.960526
recall_class_1,0.744898
f1_class_1,0.839080
roc_auc,0.961546
average_precision,0.864538



Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.96      0.74      0.84        98

    accuracy                           1.00     56962
   macro avg       0.98      0.87      0.92     56962
weighted avg       1.00      1.00      1.00     56962



## Home Credit Default Model

`HomeCreditRandomForest` uses only `hc_data.pkl` and writes its model to `models/home_credit_random_forest.joblib`. Its metrics, predictions, report, and plots are isolated under `outputs/home_credit/`.

In [9]:
home_credit_metrics = home_credit_agent.fit_evaluate_save()

Training home_credit_random_forest: 246,008 rows, 79 features, class_weight=balanced_subsample
Saved complete pipeline: c:\Users\Mer_Hass\OneDrive\Desktop\randomgorest\credit-scoring-project\models\home_credit_random_forest.joblib
Saved evaluation artifacts: c:\Users\Mer_Hass\OneDrive\Desktop\randomgorest\credit-scoring-project\outputs\home_credit


,score
accuracy,0.919256
precision_class_1,0.333333
recall_class_1,0.000201
f1_class_1,0.000403
roc_auc,0.739740
average_precision,0.217212



Classification report:
               precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.33      0.00      0.00      4965

    accuracy                           0.92     61503
   macro avg       0.63      0.50      0.48     61503
weighted avg       0.87      0.92      0.88     61503



In [11]:
def predict_fraud(applicant_features):
    """Predict credit-card fraud; return one JSON-ready dict or a list of dicts."""
    return fraud_agent.predict_records(applicant_features)


def predict_home_credit(applicant_features):
    """Predict Home Credit payment difficulty; return JSON-ready output."""
    return home_credit_agent.predict_records(applicant_features)


fraud_example = predict_fraud(fraud_agent.get_split()[0].iloc[[0]])
home_credit_example = predict_home_credit(home_credit_agent.get_split()[0].iloc[[0]])

required_prediction_keys = {
    "agent_name", "dataset_type", "predicted_class", "risk_score", "model_name"
}
for prediction in (fraud_example, home_credit_example):
    assert required_prediction_keys.issubset(prediction)
    json.dumps(prediction, allow_nan=False)
    print(prediction)

assert fraud_example["dataset_type"] == "credit_card_fraud"
assert home_credit_example["dataset_type"] == "home_credit_default"

{'agent_name': 'Agent 2 - Random Forest', 'dataset_type': 'credit_card_fraud', 'predicted_class': 0, 'risk_score': 0.0, 'model_name': 'fraud_random_forest'}
{'agent_name': 'Agent 2 - Random Forest', 'dataset_type': 'home_credit_default', 'predicted_class': 0, 'risk_score': 0.023333333333333334, 'model_name': 'home_credit_random_forest'}


## Multi-Agent Prediction Contract

`predict_fraud(applicant_features)` and `predict_home_credit(applicant_features)` accept a feature mapping, a DataFrame, or a batch of records. A single record returns one dictionary; a batch returns a list of dictionaries. Every result contains `agent_name`, `dataset_type`, `predicted_class`, `risk_score` (the estimated score for class 1), and `model_name`, using JSON-compatible Python scalar values. The repository currently has no orchestrator or existing prediction schema, so these functions establish a small standalone contract without changing other agents.

Input features must match that dataset's training feature schema. For Home Credit's 13 categorical fields, supply the numeric category codes used in `hc_data.pkl`; original string-to-code mappings are not present in the pickle. The risk score is not a calibrated lending probability. Home Credit's default 0.50 class threshold produced very low class-1 recall in the current held-out report; review the saved precision-recall curve and select an operational threshold using business costs and validation data before deployment.